# OLMo-2 1B Initial Exploration

This notebook serves as an interactive environment to load the OLMo-2 1B model, test tokenization, run sample inferences, and analyze telemetry logs.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import yaml
import torch

from src.inference import OLMoInferenceEngine
from src.evaluate import OLMoEvaluator
from src.utils import setup_logger, get_hardware_telemetry

## 1. Environment & Telemetry Verification

In [ ]:
logger = setup_logger()
telemetry = get_hardware_telemetry()
print(f"Active Device: {telemetry['device']}")
print(f"Available CUDA Devices: {telemetry['cuda_device_count']}")
if telemetry['cuda_device_count'] > 0:
    print(f"CUDA Device Name: {telemetry['cuda_device_name']}")

## 2. Load YAML Config and Serving Engine

Ensure you have downloaded and converted the model weights into `artifacts/OLMo-2-0425-1B-FP16` or point the config to your raw weights.

In [ ]:
config_path = "../configs/inference_fp16.yaml"
if not os.path.exists(config_path):
    # Fallback to local execution if running from root
    config_path = "configs/inference_fp16.yaml"

with open(config_path, "r") as f:
    config = yaml.safe_load(f)

model_path = config['model']['path']
print(f"Configured Model Path: {model_path}")

if os.path.exists(model_path) and any(f.endswith('.safetensors') or f.endswith('.bin') for f in os.listdir(model_path)):
    engine = OLMoInferenceEngine(config_path=config_path)
    print("Model loaded successfully!")
else:
    print(f"[!] Model weights not found in {model_path}. Please run download.py first.")

## 3. Interactive Generation

Generate text and inspect speed metrics.

In [ ]:
if 'engine' in locals():
    prompt = "To achieve artificial general intelligence, we must first master"
    output = engine.generate(prompt=prompt)
    print(f"Prompt: {prompt}\n")
    print(f"Generated Output:\n{output['generated_text']}\n")
    print(f"Generation speed: {output['tokens_per_sec']:.2f} tok/s")

## 4. Run Standard Metrics

Measure Perplexity (PPL) on a short baseline test phrase.

In [ ]:
if 'engine' in locals():
    evaluator = OLMoEvaluator(engine.model, engine.tokenizer, engine.device)
    sample_text = "Standardized evaluation protocols are necessary for reproducible deep learning experiments."
    ppl = evaluator.calculate_perplexity(sample_text)
    print(f"Baseline Perplexity: {ppl:.4f}")